# US 5.6 -- Multi-Round Convergence Analysis

Active domain adaptation runs over multiple rounds.  Each round adds more
labeled target samples.  The key questions are:

- **How fast does performance improve** with more labels? (learning curve)
- **Which strategy works best** for our domain? (comparison)
- **When should we stop** labeling? (stopping criterion)

This notebook provides the tools to answer all three questions.

## What you will learn

1. How to plot learning curves (F1 vs number of labels)
2. How to compare selection strategies on the same plot
3. How to apply a stopping criterion to decide when labeling is done

## CLI equivalent

```bash
udm-epic5 analyze \
    --results-dir results/ \
    --output results/convergence.png
```

## Prerequisites

- Python 3.12, matplotlib
- Install: `uv pip install -e ".[epic5]"`
- Results from multiple active learning rounds

---
## 1. Learning Curves: F1 vs Number of Labels

A learning curve shows how the target-domain F1 score improves as we add
more labeled samples across rounds.  The x-axis is the cumulative number
of labeled target samples; the y-axis is F1 on the target evaluation set.

The shape of the curve tells us:
- **Steep initial rise**: the first labels are very informative.
- **Plateau**: diminishing returns -- additional labels help less.
- **Gap to fully-supervised**: how much room remains.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from udm_epic5.analysis.convergence import learning_curve, compare_strategies, stopping_criterion

In [ ]:
# Simulate multi-round results for different strategies
# In practice, these come from actual training runs stored in results/

np.random.seed(42)

# Budget per round: 20 samples each
rounds = np.arange(0, 8)  # 0 = no target labels (DANN baseline), 1-7 = rounds
n_labels = rounds * 20     # cumulative labels: 0, 20, 40, ..., 140

# Simulated F1 scores for each strategy
# Each row: F1 at [0, 20, 40, 60, 80, 100, 120, 140] labels
def sim_curve(base, ceiling, rate, noise=0.01):
    """Simulate a logarithmic learning curve."""
    f1 = base + (ceiling - base) * (1 - np.exp(-rate * n_labels))
    f1 += np.random.normal(0, noise, len(n_labels))
    return np.clip(f1, 0, 1)

strategies = {
    'Random':      sim_curve(0.72, 0.84, 0.015, noise=0.015),
    'Uncertainty': sim_curve(0.72, 0.87, 0.020, noise=0.012),
    'Diversity':   sim_curve(0.72, 0.86, 0.018, noise=0.012),
    'Combined':    sim_curve(0.72, 0.89, 0.025, noise=0.010),
}

# Fully supervised upper bound (all target data labeled)
fully_supervised_f1 = 0.93

print("Simulated learning curve data:")
print(f"{'Labels':>8}", end='')
for name in strategies:
    print(f"  {name:>12}", end='')
print()
for i, nl in enumerate(n_labels):
    print(f"{nl:8d}", end='')
    for name, curve in strategies.items():
        print(f"  {curve[i]:12.3f}", end='')
    print()

In [ ]:
# Plot learning curves using the analysis module
# (In practice, learning_curve() reads from results files)

fig, ax = plt.subplots(figsize=(10, 6))

colors = {
    'Random': '#9E9E9E',
    'Uncertainty': '#FF5722',
    'Diversity': '#2196F3',
    'Combined': '#4CAF50',
}

for name, curve in strategies.items():
    ax.plot(n_labels, curve, 'o-', label=name, color=colors[name],
            linewidth=2, markersize=6)

# Reference lines
ax.axhline(y=fully_supervised_f1, color='black', linestyle='--', alpha=0.5,
           label=f'Fully supervised ({fully_supervised_f1:.2f})')
ax.axhline(y=strategies['Combined'][0], color='gray', linestyle=':',
           alpha=0.5, label=f'DANN baseline ({strategies["Combined"][0]:.2f})')

ax.set_xlabel('Number of labeled target samples', fontsize=12)
ax.set_ylabel('Target F1 Score', fontsize=12)
ax.set_title('Active Learning Curves by Strategy', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.65, 0.98)
fig.tight_layout()
plt.show()

print("Key observations:")
print("  - All strategies outperform random selection.")
print("  - Combined (uncertainty + diversity) converges fastest.")
print("  - ~100 labels closes most of the gap to fully supervised.")
print("  - Diminishing returns after ~80 labels.")

---
## 2. Comparing All Strategies

The `compare_strategies` function computes summary statistics across
strategies to help pick the best one.

In [ ]:
# Compute area under the learning curve (AULC)
# Higher AULC = faster convergence = better strategy

print("Strategy comparison:")
print(f"{'Strategy':>15}  {'Final F1':>9}  {'AULC':>8}  {'Labels to 0.85':>15}")
print("-" * 55)

for name, curve in strategies.items():
    final_f1 = curve[-1]
    aulc = np.trapz(curve, n_labels) / (n_labels[-1] - n_labels[0])

    # How many labels to reach F1 >= 0.85?
    labels_to_target = 'N/A'
    for i, f1 in enumerate(curve):
        if f1 >= 0.85:
            labels_to_target = str(n_labels[i])
            break

    print(f"{name:>15}  {final_f1:9.3f}  {aulc:8.3f}  {labels_to_target:>15}")

print()
print("AULC (Area Under Learning Curve) summarises overall efficiency.")
print("'Labels to 0.85' shows how quickly each strategy reaches the target F1.")

In [ ]:
# Bar chart: F1 improvement over DANN baseline at fixed budget
budget_idx = 3  # 60 labels
baseline_f1 = strategies['Combined'][0]

fig, ax = plt.subplots(figsize=(8, 5))

names = list(strategies.keys())
improvements = [strategies[n][budget_idx] - baseline_f1 for n in names]

bars = ax.bar(names, improvements, color=[colors[n] for n in names],
              edgecolor='white', linewidth=0.5)

for bar, imp in zip(bars, improvements):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'+{imp:.3f}', ha='center', va='bottom', fontsize=11)

ax.set_ylabel('F1 improvement over DANN baseline', fontsize=12)
ax.set_title(f'F1 Improvement at {n_labels[budget_idx]} Labeled Samples', fontsize=14)
ax.grid(True, alpha=0.2, axis='y')
fig.tight_layout()
plt.show()

---
## 3. Stopping Criterion

When should we stop the active learning loop?  Three common criteria:

1. **Plateau detection**: stop when the F1 improvement between rounds drops
   below a threshold (e.g., < 0.005 for two consecutive rounds).
2. **Target F1 reached**: stop when we reach a desired F1 score.
3. **Budget exhausted**: stop after a fixed number of rounds.

The `stopping_criterion` function implements all three.

In [ ]:
# Demonstrate stopping criterion with the Combined strategy
combined_curve = strategies['Combined']

print("Round-by-round analysis (Combined strategy):")
print(f"{'Round':>6}  {'Labels':>7}  {'F1':>7}  {'Delta F1':>9}  {'Stop?':>6}")
print("-" * 45)

for i in range(len(combined_curve)):
    delta = combined_curve[i] - combined_curve[i-1] if i > 0 else 0.0
    # Plateau: delta < 0.005 for two consecutive rounds
    should_stop = False
    if i >= 2:
        prev_delta = combined_curve[i-1] - combined_curve[i-2]
        should_stop = (delta < 0.005) and (prev_delta < 0.005)

    print(f"{i:6d}  {n_labels[i]:7d}  {combined_curve[i]:7.3f}  "
          f"{delta:+9.3f}  {'YES' if should_stop else 'no':>6}")

print()
print("The stopping criterion detects when additional labels provide")
print("diminishing returns, saving annotation effort.")

In [ ]:
# Visualise the stopping point on the learning curve
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(n_labels, combined_curve, 'o-', color='#4CAF50', linewidth=2,
        markersize=8, label='Combined strategy')

# Mark the deltas
for i in range(1, len(combined_curve)):
    delta = combined_curve[i] - combined_curve[i-1]
    mid_x = (n_labels[i-1] + n_labels[i]) / 2
    mid_y = (combined_curve[i-1] + combined_curve[i]) / 2
    ax.annotate(f'{delta:+.3f}', xy=(mid_x, mid_y),
                fontsize=9, ha='center', va='bottom', color='#666')

# Find stopping point (first round where plateau is detected)
stop_round = None
for i in range(2, len(combined_curve)):
    d1 = combined_curve[i] - combined_curve[i-1]
    d0 = combined_curve[i-1] - combined_curve[i-2]
    if d1 < 0.005 and d0 < 0.005:
        stop_round = i
        break

if stop_round is not None:
    ax.axvline(x=n_labels[stop_round], color='red', linestyle='--', alpha=0.7,
               label=f'Stopping point ({n_labels[stop_round]} labels)')
    ax.scatter([n_labels[stop_round]], [combined_curve[stop_round]],
               s=200, c='red', zorder=10, marker='X')

ax.axhline(y=fully_supervised_f1, color='black', linestyle='--', alpha=0.3,
           label=f'Fully supervised ({fully_supervised_f1:.2f})')

ax.set_xlabel('Number of labeled target samples', fontsize=12)
ax.set_ylabel('Target F1 Score', fontsize=12)
ax.set_title('Learning Curve with Stopping Criterion', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.65, 0.98)
fig.tight_layout()
plt.show()

if stop_round is not None:
    saved = n_labels[-1] - n_labels[stop_round]
    print(f"Stopping at round {stop_round} ({n_labels[stop_round]} labels) ")
    print(f"saves {saved} labels with minimal F1 loss.")
else:
    print("No stopping point detected -- all rounds show improvement.")

---
## 4. Per-Domain Convergence

Different target domains may converge at different rates.  It is useful to
plot learning curves per domain to identify which sites need more labels.

In [ ]:
# Simulated per-domain results (Combined strategy)
per_domain = {
    'Malaysia': sim_curve(0.74, 0.90, 0.025, noise=0.010),
    'Mexico':   sim_curve(0.71, 0.87, 0.020, noise=0.012),
    'China':    sim_curve(0.68, 0.84, 0.018, noise=0.015),
}

fig, ax = plt.subplots(figsize=(10, 6))

domain_colors = {'Malaysia': '#FF5722', 'Mexico': '#2196F3', 'China': '#9C27B0'}

for domain, curve in per_domain.items():
    ax.plot(n_labels, curve, 'o-', label=domain, color=domain_colors[domain],
            linewidth=2, markersize=6)

ax.axhline(y=0.85, color='gray', linestyle='--', alpha=0.5,
           label='Target F1 = 0.85')

ax.set_xlabel('Number of labeled target samples', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Domain Convergence (Combined Strategy)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.60, 0.98)
fig.tight_layout()
plt.show()

print("Observations:")
print("  - Malaysia converges fastest (closest to source distribution).")
print("  - China needs more labels (largest domain gap).")
print("  - Consider allocating more budget to harder domains.")

---
## 5. CLI Usage

```bash
# Generate convergence analysis plots
udm-epic5 analyze \
    --results-dir results/ \
    --output results/convergence.png

# Compare specific strategies
udm-epic5 analyze \
    --results-dir results/ \
    --strategies random,uncertainty,diversity,combined \
    --output results/strategy_comparison.png

# Check stopping criterion
udm-epic5 analyze \
    --results-dir results/ \
    --check-stopping \
    --threshold 0.005 \
    --patience 2
```

The CLI reads results from the `results/` directory (one JSON per round)
and generates publication-quality plots.

---
## Summary

- Learning curves show F1 vs number of labels -- the key metric for active learning.
- Combined (uncertainty + diversity) consistently outperforms single-criterion strategies.
- The stopping criterion detects diminishing returns and saves labeling effort.
- Per-domain analysis reveals which target sites need more labels.
- Use `udm-epic5 analyze` for automated convergence reporting.

### Epic 5 complete workflow

```
1. Train DANN (Epic 4)           -> baseline model
2. Compute uncertainty (US 5.1)  -> entropy scores
3. Diversity selection (US 5.2)  -> diverse subset
4. Combined selection (US 5.3)   -> final selection
5. Human labeling (US 5.4)       -> target masks
6. Active DANN training (US 5.5) -> improved model
7. Convergence check (US 5.6)    -> stop or repeat from step 2
```